In [23]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *
from functions.repeated_cv import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import ComponentwiseGradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [24]:
dataset = pd.read_csv("../../data/data_final.csv")

diagnosis_variables = []

with open("../../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [25]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [26]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [27]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [28]:
param_grid = {
    'learning_rate': [0.05, 0.1],
    'n_estimators': [1000],
    'dropout_rate': [0.0],
    'n_controls_per_case': [1, 5, 10, 15]
    }

metrics = ['c-index', 'c-index-ipcw', 'int-brier-score']

In [29]:
results_df, summary_df, best_params = ensemble_repeated_grid_search(
    ComponentwiseGradientBoostingSurvivalAnalysis, x_train, y_train, param_grid, metrics,
    model_name='GBM Model',
    scale_data=False, scaler_type='standard',
    random_state=None, n_jobs=None,
    n_controls_per_case=None, 
    dataset=subsampling,
    feature_columns=feature_columns,
    event_col='Regional LN recurrence',
    time_col='Survival days',
    n_splits=4, n_repeats=5, base_random_state=42,
    verbose=True
)


GBM MODEL REPEATED GRID SEARCH
Model: ComponentwiseGradientBoostingSurvivalAnalysis
Scaling: No
Subsampling: No
Metrics: ['c-index', 'c-index-ipcw', 'int-brier-score']
Parameter Grid:
  - learning_rate: [0.05, 0.1]
  - n_estimators: [1000]
  - dropout_rate: [0.0]
  - n_controls_per_case: [1, 5, 10, 15]
Total hyperparameter combinations: 8
Folds per repetition: 4
Repetitions: 5
Total evaluations: 160

[1/8] Testing params: {'learning_rate': 0.05, 'n_estimators': 1000, 'dropout_rate': 0.0}, n_controls_per_case=1
  c-index: 0.8076 ± 0.0801
  c-index-ipcw: 0.7677 ± 0.0861
  int-brier-score: 0.0538 ± 0.0110
[2/8] Testing params: {'learning_rate': 0.05, 'n_estimators': 1000, 'dropout_rate': 0.0}, n_controls_per_case=5
  Elapsed: 0.4min, Est. remaining: 2.6min
  c-index: 0.8146 ± 0.0887
  c-index-ipcw: 0.7385 ± 0.1682
  int-brier-score: 0.0184 ± 0.0036
[3/8] Testing params: {'learning_rate': 0.05, 'n_estimators': 1000, 'dropout_rate': 0.0}, n_controls_per_case=10
  Elapsed: 0.7min, Est. rema

In [31]:
summary_df.to_csv('../../results/repeated_sub/gbm_wise_sub_1.csv', index=False)